# Pre-entrega 3

Importar librerías

In [1]:
import os
import glob
import json
import hashlib
import logging
import shutil
import dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()


/var/folders/qh/hxdbc3t54754snk_wm1ltbhw0000gn/T/ipykernel_13262/2019494717.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Crear documentos

In [2]:
## Creación de archivos de texto
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


os.makedirs("data", exist_ok=True)

texto1 = """
Convivencia Privada, Destino de las Unidades y Régimen de Tenencia de Mascotas
El régimen de propiedad exclusiva y convivencia dentro del Edificio Riesco 6540 delimita de manera estricta los derechos, facultades y prohibiciones que recaen sobre los propietarios, arrendatarios y ocupantes a cualquier título respecto al uso de sus departamentos, bodegas y espacios de uso común. De conformidad con el Artículo Segundo y Tercero del Reglamento de Copropiedad, cada titular ostenta el dominio absoluto sobre su unidad privada y comparte la calidad de comunero sobre los bienes destinados al servicio y conservación general, quedando plenamente sometido a las disposiciones de este reglamento y de la Ley N° 19.537 sobre Copropiedad Inmobiliaria. El cumplimiento de estas normas es obligatorio tanto para los adquirentes originarios como para los sucesores en el dominio y para cualquier tercero a quien se le ceda la tenencia o goce del inmueble.
Tenencia y Regulación de Animales Domésticos o Mascotas
La tenencia de animales en el condominio se encuentra regulada en el Artículo Séptimo Transitorio, precepto que establece una distinción temporal y operativa crucial para los residentes. Según esta disposición, la presencia de animales domésticos o mascotas está expresamente permitida durante la etapa inicial del edificio, esto es, mientras se procede a la constitución formal del Comité de Administración. Una vez que dicho comité entre en funciones, corresponderá a este organismo, en acuerdo con la asamblea de copropietarios, fijar las reglas definitivas y particulares que regirán la mantención, tenencia y responsabilidades derivadas de las mascotas. La empresa inmobiliaria vendedora queda totalmente liberada de cualquier conflicto o decisión adoptada por la comunidad respecto a esta materia.
Sin perjuicio de la normativa que dicte con posterioridad el Comité de Administración, el reglamento fija dos obligaciones mínimas de carácter permanente e inexcusable para todo ocupante:
	1.	Prohibición de circulación libre o sin sujeción: Queda estrictamente prohibido el tránsito de animales sueltos o sin su correspondiente collar o correa en cualquiera de los espacios comunes del edificio, incluidos pasillos, vestíbulos, jardines, accesos y ascensores.
	2.	Obligación inmediata de aseo y saneamiento: Los dueños o tenedores de las mascotas están obligados a limpiar y sanitizar de forma inmediata los espacios comunes en caso de que los animales llegasen a ensuciar, botar desperdicios o generar manchas en dichos recintos.
El incumplimiento de cualquiera de estas normas mínimas sobre mascotas constituye una infracción directa sujeta a la aplicación de las sanciones y multas previstas en el régimen general de penalizaciones del Artículo Noveno.
Destino Exclusivo Habitacional y Prohibición Comercial
El Artículo Octavo consagra que cada unidad de departamento debe ser utilizada en forma ordenada y tranquila, encontrándose destinada exclusivamente a fines de vivienda o habitación, según las autorizaciones legales y las ordenanzas municipales vigentes. Está terminantemente prohibido desviar las unidades hacia cualquier otra finalidad. Esta restricción implica la prohibición absoluta de comercializar productos, almacenar mercaderías con fines de lucro o ejercer actividades comerciales, profesionales o industriales abiertas al público al interior de los departamentos o de las dependencias comunes. Cualquier cambio de destino requiere el cumplimiento estricto del Artículo Décimo Primero: que el uso esté admitido en el instrumento de planificación territorial comunal, la aprobación previa de la Dirección de Obras Municipales de Las Condes y el acuerdo formal de la asamblea de copropietarios.
En idéntico sentido, las bodegas subterráneas solo pueden ser destinadas al almacenamiento o depósito de enseres correspondientes al menaje habitual de una casa habitación. El reglamento prohíbe taxativamente guardar mercadería comercial, productos peligrosos o bienes de terceros ajenos a la vida cotidiana. Asimismo, para prevenir deterioros por filtraciones o humedad, los ocupantes deben ubicar los objetos almacenados a una distancia mínima de diez centímetros por sobre el nivel del suelo de la bodega.
Normas de Convivencia, Emisiones de Humo y Ruido
La tranquilidad y el descanso de los vecinos configuran una obligación esencial contemplada en las letras c), e) y q) del Artículo Octavo. Los residentes deben abstenerse de ejecutar actos o generar ruidos que perturben la convivencia pacífica, especialmente en los horarios tradicionalmente dedicados al reposo nocturno. Está prohibido hacer funcionar equipos de sonido, instrumentos musicales o televisores a volúmenes estridentes en cualquier momento del día o la noche, así como mantener conversaciones bulliciosas o formar grupos en pasillos, jardines y escaleras.
Respecto al consumo de tabaco, rige una prohibición total de fumar en todas las áreas de dominio común, accesos, circulaciones, subterráneos y bodegas. Si bien no está vedado fumar al interior de las unidades privadas, el copropietario u ocupante tiene el deber de evitar que las emanaciones se propaguen hacia los departamentos colindantes o hacia los pasillos. En particular, se prohíbe expresamente fumar en piezas, baños o cocinas que cuenten con sistemas de ventilación mediante shafts o ductos colectivos sin ventilación natural directa, debido a que el tiro de estos conductos arrastra el humo hacia otras viviendas del edificio. La administración cuenta con atribuciones expresas para fiscalizar y determinar si un ocupante está infringiendo esta limitación.
Procedimiento Sancionatorio y Régimen de Multas
Cualquier transgresión a las obligaciones descritas en el Artículo Octavo o en los artículos transitorios activa el procedimiento disciplinario regulado en el Artículo Noveno:
⚬	Monto de la infracción: El Comité de Administración o el Administrador están facultados para aplicar multas a beneficio de la comunidad equivalentes a dos Unidades Tributarias Mensuales (UTM). Estas multas comienzan a devengarse desde la fecha en que se notifica por carta certificada la infracción y continúan generándose de forma sucesiva hasta la subsanación material del hecho.
⚬	Reincidencia: Si se comete la misma infracción dentro de los seis meses posteriores a la resolución sancionatoria —incluso si los afectados directos fueren distintos residentes—, el valor de la sanción se elevará automáticamente al doble, es decir, a cuatro UTM.
⚬	Responsabilidad solidaria y cobro: El propietario de la unidad y el infractor directo (como un arrendatario u ocupante) son solidariamente responsables del pago de las multas cursadas y de las indemnizaciones por perjuicios. El importe recaudado por multas no ingresa al gasto corriente, sino que se integra íntegramente al Fondo Común de Reserva.
⚬	Derecho de defensa y sede judicial: El afectado puede reclamar de la multa ante el Juez de Policía Local competente dentro de un plazo de quince días corridos contados desde el despacho de la carta certificada enviada por medio de un Ministro de Fe.
"""

texto2 = """
Espacios Comunes, Áreas Recreativas, Salón Multiuso y Uso de la Piscina
El Edificio Riesco 6540 integra una variedad de dependencias e instalaciones colectivas concebidas para el esparcimiento, la seguridad, la administración y la vida comunitaria de los propietarios y moradores legítimos. De acuerdo con el Artículo Primero y el Artículo Cuarto del Reglamento de Copropiedad, se reconocen como bienes comunes necesarios e inseparables del dominio individual el terreno, los accesos, los vestíbulos, pasillos, ascensores, escaleras, gimnasio, guardería o extensión de gimnasio, sala de reuniones, sala de enfermería, sala de encomiendas, salón multiuso y piscina del primer piso, junto con sus recintos complementarios. Cada uno de estos bienes se encuentra sometido a un riguroso régimen de utilización prudente que busca evitar el deterioro de la infraestructura y el perjuicio al uso legítimo de los demás copropietarios.
Régimen Integral de la Piscina del Condominio
La piscina principal del edificio, ubicada a nivel del primer piso, es un bien de uso y dominio común calificado como recreativo conforme a las letras d) del Artículo Cuarto y w) del Artículo Octavo. Su funcionamiento operativo y las condiciones de admisión se rigen bajo los siguientes parámetros reglamentarios:
	1.	Titularidad y Exclusividad: Las instalaciones de la piscina están reservadas al uso exclusivo de los copropietarios del condominio y de las personas que ocupen legítimamente las unidades como arrendatarios. No constituye un espacio de libre concurrencia para personas ajenas a la comunidad.
	2.	Definición de Temporadas y Horarios: La piscina no permanece abierta de forma indiscriminada a lo largo del año. El reglamento ordena que la época del año en que estará habilitada y los horarios diarios de funcionamiento deben ser definidos conjuntamente por los copropietarios y la administración en la primera asamblea ordinaria que celebre la comunidad. Las resoluciones adoptadas en dicha instancia son de carácter obligatorio para todos los residentes.
	3.	Publicidad y Deber de Información: Corresponde al Administrador del edificio exhibir de manera permanente, en un lugar visible en las inmediaciones del recinto de la piscina y en las vitrinas de conserjería, el cuadro completo de horarios de apertura, cierre y las normas disciplinarias de higiene y seguridad que rijan su uso.
	4.	Régimen Obligatorio para Personas no Copropietarias (Visitas): Si un copropietario o residente desea ingresar a la piscina acompañado de personas que no residan en el edificio, debe realizar una solicitud previa por escrito al Administrador. El Administrador mantiene la facultad discrecional de autorizar o denegar el permiso en función del aforo, determinando el rango horario exacto y el número máximo de acompañantes autorizados. Además, este ingreso de visitas se encuentra condicionado al pago anticipado de la tarifa especial que fije el Comité de Administración para estos fines.
	5.	Instalaciones Sanitarias Anexas: La infraestructura asociada a la piscina del primer piso incluye un lavapiés de acceso y recintos de baño independientes para usuarios de la piscina. El mantenimiento técnico de los equipos de bombeo, filtrado y recirculación de agua es un gasto común ordinario que la administración debe resguardar mediante contratos periódicos suscritos con personal técnico autorizado.
Condiciones y Reglas de Utilización del Salón Común (Multiuso)
El salón común, situado en el primer piso del inmueble, está destinado al esparcimiento y a la celebración de reuniones o eventos particulares organizados por los copropietarios. El Artículo Cuarto y la letra w) del Artículo Octavo imponen condiciones operativas estrictas para evitar desórdenes, ruidos molestos o daños materiales:
⚬	Capacidad Máxima y Aforo: El salón está diseñado estructuralmente para albergar una capacidad máxima de veinte personas. Está terminantemente prohibido sobrepasar este límite de asistentes, con la única excepción de las reuniones formales de Asambleas de Copropietarios.
⚬	Límite Horario Nocturno: Cuando un residente arriende o solicite el salón para un evento privado, la actividad no puede prolongarse más allá de la una de la madrugada (01:00 AM). Cualquier modificación extraordinaria a este horario solo puede emanar de un acuerdo expreso del Comité de Administración.
⚬	Obligaciones de Higiene y Entrega: Quien utilice el salón asume el compromiso ineludible de colaborar con el aseo, limpieza y orden inmediato tras el término de la actividad, debiendo reintegrar las dependencias, baños y mobiliario en idéntico estado de conservación al recibido.
⚬	Ingresos y Reparaciones: Los dineros que perciba el condominio por concepto de arriendo del salón multiuso o por multas aplicadas por su uso incorrecto constituyen ingresos de la comunidad. El Administrador puede cursar sanciones pecuniarias contra quienes infrinjan el orden o causen destrozos, según el tarifario fijado por los copropietarios.
Equipamiento Comunitario Adicional del Primer Piso
El primer piso concentra los servicios primarios del edificio. Entre estos se encuentran:
⚬	Gimnasio y Guardería/Extensión de Gimnasio: Diseñados para la salud y recreación de los ocupantes, sujetos a las normas disciplinarias de conservación y orden que emita el Comité de Administración.
⚬	Sala de Enfermería o Primeros Auxilios: Espacio de atención primaria para contingencias sanitarias o emergencias menores.
⚬	Sala de Encomiendas y Sala de Reuniones: Recintos destinados a optimizar la entrega segura de paquetes y la coordinación institucional interna.
⚬	Dependencias de Conserjería: Baño propio y kitchenette para garantizar la adecuada permanencia y dignidad del personal laboral del condominio.
Preservación del Libre Tránsito y Prohibición de Ocupación
Los copropietarios u ocupantes tienen derecho a servirse de los bienes comunes usándolos de manera prudencial en su destino ordinario. Las letras h), r) y s) del Artículo Octavo prohíben estrictamente ocupar u obstaculizar pasillos, halls de ascensores, vestíbulos, escaleras y veredas con muebles, alfombras, bicicletas, choapinos, cajas o cualquier tipo de objeto privado. Nadie puede tomar posesión exclusiva de un área común exterior a su departamento. La administración está expresamente facultada para retirar de inmediato cualquier bulto u obstáculo depositado indebidamente en las vías de circulación por cuenta, costo y riesgo del infractor.
"""

texto3 = """
Asados, Seguridad Contra Incendios, Balcones, Terrazas y Unidades Exclusivas
La seguridad estructural, la prevención de siniestros y la preservación estética y armónica del Edificio Riesco 6540 imponen límites rigurosos respecto a las fuentes de calor, el manejo de fuego, la evacuación de humos y la intervención de las fachadas exteriores y terrazas. Las normas del reglamento abordan con precisión técnica los riesgos vinculados a incendios, malos olores, desprendimiento de residuos e intervenciones no autorizadas en las envolventes arquitectónicas del edificio.
Regulación Exhaustiva sobre Asados y Tipo de Parrillas
El régimen para la preparación de alimentos al aire libre, encendido de brasas y utilización de parrillas está taxativamente determinado en el Artículo Octavo, letra x). Dicha disposición establece la siguiente jerarquía de prohibiciones y permisos según la ubicación y el tipo de unidad:
	1.	Prohibición General: Queda formal y estrictamente prohibido instalar o utilizar parrillas a carbón en los departamentos estándar (pisos segundo al décimo quinto) y en todas las áreas o dependencias comunes del condominio.
	2.	Excepción para Departamentos del Primer Piso (Parrillas a Gas): Los departamentos situados en el primer piso que gozan de la asignación de polígonos de uso y goce exclusivo sobre jardines (específicamente departamentos 101 y 104) están autorizados a utilizar únicamente parrillas a gas. Mantienen la prohibición de encender carbón o leña.
	3.	Excepción para Piso Retirado y Planta Cubierta (Parrillas a Carbón): Los departamentos que cuentan con asignaciones de uso y goce exclusivo en el piso retirado y en la planta cubierta (correspondientes a los departamentos 1502, 1503, 1601 y 1604) se encuentran expresamente liberados de la prohibición general, pudiendo encender y operar parrillas a carbón dentro de sus polígonos asignados.
Esta regulación tiene como fundamento la prevención del riesgo de incendios y la necesidad de evitar la propagación ascendente de humos y cenizas a través de las fachadas hacia las ventanas y loggias de las unidades superiores.
Prevención de Incendios y Materiales Inflamables
De acuerdo con las letras c), f) y m) del Artículo Octavo, los residentes tienen estrictamente prohibido acumular, almacenar o manipular dentro de sus departamentos, bodegas, estacionamientos o bienes comunes depósitos de materias explosivas, sustancias inflamables, corrosivas, malolientes o tóxicas que comprometan la seguridad y habitabilidad de la comunidad.
En lo relativo a los ductos y tolvas de basura, está terminantemente prohibido evacuar papeles encendidos, botellas, tarros, piedras, cartones voluminosos o elementos inflamables que pudieran generar incendios en los shafts o salas de compactación. El edificio dispone de un sistema de prevención y combate compuesto por red húmeda, red seca y equipos de detección de alarmas que constituyen bienes de dominio común sujetos a contratos de mantención técnica obligatoria gestionados por la administración y el Comité de Administración de acuerdo con el Artículo Trigésimo Primero.
Régimen de Terrazas, Balcones y Preservación de Fachadas
La apariencia exterior y la habitabilidad de las terrazas se encuentran salvaguardadas en las letras b), i), k), l) y u) del Artículo Octavo:
⚬	Tendido de Ropa: Está estrictamente prohibido tender prendas, toallas, sábanas o ropa en barandas, terrazas, balcones o ventanas que den al exterior. El secado de prendas solo puede efectuarse en las loggias, lavaderos o recintos interiores especialmente adaptados al efecto.
⚬	Cortinas y Toldos: Para resguardar la uniformidad cromática y estética del conjunto, las cortinas interiores que se orienten hacia el exterior deben ser obligatoriamente de color blanco. En las terrazas de cada departamento solo se permite instalar un tipo específico de protección solar: toldo vertical de color blanco modelo droppy cabrio, marca LUXAFLEX o similar. Está prohibido instalar lonas de otros colores, adhesivos reflectantes, mallas de tonalidades diversas o elementos que modifiquen la línea del edificio.
⚬	Prohibición de Baldeo y Control de Riego: Queda formalmente prohibido baldear los pisos de las terrazas o regar plantas mediante mangueras de presión. El riego de jardineras y maceteros debe hacerse con el debido cuidado para evitar escurrimientos o filtraciones hacia los departamentos inferiores que ensucien ventanales, muros y fachadas. La limpieza periódica de la fachada general y de los cristales no autolimpiables corresponde a la comunidad de copropietarios mediante gastos comunes.
Polígonos de Uso y Goce Exclusivo en Cubiertas y Jardines
El Artículo Cuarto Transitorio describe los derechos y limitaciones sobre las áreas comunes entregadas en uso y goce exclusivo:
⚬	Jardines de Primer Piso (Polígonos 1 y 2): Asignados a los departamentos 101 y 104. Son mantenidos en su totalidad por los jardineros de la comunidad. A los propietarios les está prohibido alterar el paisajismo sin la autorización expresa y unánime de todos los copropietarios del condominio, así como instalar cercos o regadíos suplementarios. Los consumos de agua y electricidad de estos jardines se registran directamente en los remarcadores individuales de los departamentos 101 y 104.
⚬	Terrazas y Cubiertas Superiores (Polígonos 3, 4, 5 y 6): Asignados a los departamentos 1502, 1503, 1601 y 1604. Los departamentos 1502 y 1503 cuentan en sus polígonos con lavaplatos y equipos de jacuzzi, cuyo gasto de electricidad y agua fría/caliente se registra en sus respectivos remarcadores. A su vez, los departamentos 1601 y 1604 cuentan en la planta cubierta con piscinas de uso privado. Las cuentas de llenado y filtrado de estas piscinas y la iluminación de estos polígonos se cargan exclusivamente al consumo individual de los departamentos 1601 y 1604. Todos los titulares de polígonos deben pagar una contribución periódica adicional consignada en el Anexo II, la que se integra al Fondo de Reserva del edificio.
"""

texto4 = """
Obras, Alteraciones Estéticas, Mudanzas, Estacionamientos y Gastos Comunes
La administración de un edificio de alta densidad exige mecanismos técnicos y financieros que resguarden la vida útil del inmueble, el mantenimiento de los servicios básicos y la seguridad vial de las áreas de estacionamiento. El Reglamento de Copropiedad del Edificio Riesco 6540 articula procedimientos claros para la realización de obras privadas, el traslado de carga mediante ascensores, la asignación de estacionamientos de visitas y la recaudación de los gastos comunes y fondos de reserva.
Normas sobre Reformas Interiores y Prohibición de Modificar Pisos
Las modificaciones al interior de los departamentos se encuentran acotadas para evitar daños estructurales o perturbaciones acústicas a las viviendas vecinas. La letra z) del Artículo Octavo contiene una prohibición perentoria: está prohibido cambiar el piso flotante entregado originalmente por entablados de madera, parquet, baldosas o porcelanatos. Esta restricción busca mitigar la transmisión del ruido por impacto a las losas inferiores. La única excepción a esta regla aplica a los departamentos que no tienen otra vivienda habitacional inmediatamente abajo, individualizados exclusivamente como los departamentos 101, 104, 202 y 203.
Asimismo, las letras b) y g) del Artículo Octavo prohíben demoler o modificar tabiques soportantes, alterar muros divisorios, perforar shafts colectivos para instalar equipos de aire acondicionado o extractores ajenos a la ingeniería original, o sustituir las ventanas por materiales, colores o cristales distintos a los instalados por la constructora.
Clasificación de Faenas y Horarios de Trabajo
El Artículo Décimo Transitorio clasifica las obras e intervenciones que se ejecuten dentro de las unidades o espacios comunes en tres categorías operativas:
	1.	Reparaciones de Emergencia: Aquellas provocadas por fallas fortuitas, roturas de cañerías, inundaciones, desperfectos de gas, corte del tendido eléctrico o daño en cerraduras. Su ejecución es inmediata y su horario es flexible, procurando ocasionar el menor impacto a los vecinos. La habilitación de ascensores para estos casos es expedita y coordinada por la administración.
	2.	Reparaciones Menores o Mantenciones Simples: Incluyen trabajos de pintura de muros interiores, cambios menores en tabiques divisorios livianos o reemplazo de sanitarios y mobiliario. De requerirse el uso de ascensores para traslado de materiales o escombros, solo podrá emplearse el ascensor designado por la administración y dentro de los siguientes horarios: de lunes a viernes de 07:00 a 19:30 horas y sábados de 09:00 a 14:00 horas. Se prohíbe todo trabajo los días domingos y festivos.
	3.	Reparaciones Mayores: Obras de remodelación integral, cambio de pavimentos autorizados, reformas completas de baños o recableado eléctrico. Estas intervenciones solo pueden realizarse dentro de un mismo recinto con una periodicidad máxima de una vez cada dos años y la faena no puede exceder los ciento veinte días corridos. Operan bajo las mismas restricciones horarias de ascensores y prohibición dominical.
Protocolo de Mudanzas y Carga en Ascensores
De acuerdo con la letra w) del Artículo Octavo, los ascensores están concebidos exclusivamente para el transporte de personas. El acarreo de bultos pesados, materiales de construcción y mudanzas exige una autorización previa de la administración, la cual debe colocar las fundas de protección y habilitar el elevador correspondiente. Las mudanzas y traslados de carga están autorizados únicamente en los siguientes bloques horarios:
⚬	Lunes a viernes: Entre las 07:00 y las 17:30 horas.
⚬	Sábados: Entre las 09:00 y las 14:00 horas.
⚬	Domingos y días festivos: Queda estrictamente prohibida la realización de mudanzas.
Régimen de Estacionamientos de Visitas y Circulaciones Viales
El edificio cuenta con seis estacionamientos para visitas en superficie y diez en el primer subterráneo. Las letras d), s), v) e y) del Artículo Octavo fijan las normas viales del recinto:
⚬	Límite Temporal para Copropietarios: Los estacionamientos de visitas son bienes de dominio común. Se prohíbe que los copropietarios o residentes utilicen estos estacionamientos por un lapso superior a una hora continua, restricción aplicable tanto en jornada diurna como nocturna.
⚬	Prohibición de Lavado y Baldeo: Está estrictamente prohibido baldear el pavimento de los estacionamientos o utilizar las mangueras y estacionamientos de visitas para lavar vehículos particulares.
⚬	Taxis y Vehículos de Pasajeros: El transporte comercial de pasajeros que acuda a buscar o dejar residentes solo puede permanecer estacionado por un lapso máximo de media hora.
⚬	Enajenación y Arriendo: Está prohibido vender o arrendar estacionamientos o bodegas a personas o terceros ajenos al condominio. Se faculta a la administración para retirar cualquier vehículo o bulto que impida la normal circulación en las vías subterráneas.
Gastos Comunes, Prorrateo Centralizado y Corte de Suministro
Los Artículos Vigésimo Segundo, Vigésimo Tercero y Vigésimo Cuarto norman la solvencia económica comunitaria:
⚬	Plazo de Pago: Los gastos comunes deben enterarse dentro de los cinco días siguientes a la emisión de la liquidación o aviso de cobro. En caso de mora, la deuda se reajusta en Unidades de Fomento y devenga el interés máximo convencional.
⚬	Combustible para Caldera (Agua Caliente y Calefacción): El costo de combustible se cobra dividiendo su valor mensual en dos componentes: el 50% se prorratea entre todas las unidades según su alícuota de avalúo fiscal y el 50% restante se prorratea en función del consumo efectivo medido en los remarcadores individuales de cada departamento.
⚬	Suspensión de Suministro Eléctrico: El Administrador, previa autorización del Comité de Administración, está plenamente facultado para suspender o solicitar a la compañía eléctrica el corte del servicio a toda unidad cuyos titulares se encuentren morosos en el pago de dos o más cuotas de gastos comunes, sean estas continuas o discontinuas.
⚬	Fondo Común de Reserva: Se integra mediante recargos periódicos en los gastos comunes ordinarios, cobro de multas a infractores y aportes adicionales de los polígonos exclusivos de cubiertas y jardines. Está destinado a cubrir imprevistos urgentes y obras extraordinarias debidamente acordadas.
"""

documentos = {
    "convivencia_privada.txt": texto1,
    "espacios_comunes.txt": texto2,
    "asados.txt": texto3,
    "obras_mudanzas.txt": texto4,
}

for nombre, contenido in documentos.items():
    with open(os.path.join("data", nombre), "w", encoding="utf-8") as f:
        f.write(contenido.strip())
        logging.info(f"Archivo {nombre} creado con éxito.")

2026-09-23 00:56:41,846 - INFO - Archivo convivencia_privada.txt creado con éxito.
2026-09-23 00:56:41,847 - INFO - Archivo espacios_comunes.txt creado con éxito.
2026-09-23 00:56:41,847 - INFO - Archivo asados.txt creado con éxito.
2026-09-23 00:56:41,848 - INFO - Archivo obras_mudanzas.txt creado con éxito.


In [3]:
## Ingesta
loader = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
original_docs = loader.load()
logging.info(f"{len(original_docs)} documentos cargados desde la carpeta 'data'.")

# Parametros de fragmentacion. Son constantes porque forman parte de la
# identidad del indice: cambiarlos invalida los vectores ya calculados.
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# Los reglamentos estan escritos como secciones con titulo y viñetas "\u26ac".
# Partir SOLO por tamaño hace que una norma corta (ej: "Tendido de Ropa") quede
# como cola de un chunk cuyo tema dominante es otro (basura, incendios): el
# embedding promedia todo el chunk y la norma se vuelve irrecuperable.
# Respetar los separadores del documento mantiene un tema por chunk.
SEPARATORS = ["\n\u26ac", "\n\t", "\n\n", "\n", ". ", " "]

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
)

chunks = splitter.split_documents(original_docs)

logging.info(f"{len(chunks)} fragmentos generados a partir de los documentos originales.")
print("\n--- Ejemplo de fragmento ---")
print(chunks[0].page_content[:300])
print("Metadata:", chunks[0].metadata)

2026-09-23 00:56:41,859 - INFO - 4 documentos cargados desde la carpeta 'data'.
2026-09-23 00:56:41,954 - INFO - 69 fragmentos generados a partir de los documentos originales.



--- Ejemplo de fragmento ---
Obras, Alteraciones Estéticas, Mudanzas, Estacionamientos y Gastos Comunes
La administración de un edificio de alta densidad exige mecanismos técnicos y financieros que resguarden la vida útil del inmueble, el mantenimiento de los servicios básicos y la seguridad vial de las áreas de estacionamiento
Metadata: {'source': 'data/obras_mudanzas.txt'}


In [ ]:
## Embeddings y persistencia
PERSIST_DIRECTORY = "./vectorstore"
COLLECTION_NAME = "apartment_rules"
META_PATH = os.path.join(PERSIST_DIRECTORY, "index_meta.json")

# all-MiniLM-L6-v2 es monolingue (ingles). Sobre texto en español los scores se
# aplastan (0.366 a 0.390 entre el mejor y el peor chunk) y deja de discriminar.
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


def huella_actual(data_dir: str = "data") -> dict:
    """Identidad del indice: QUE se indexo y CON QUE configuracion."""
    sha = hashlib.sha256()
    for ruta in sorted(glob.glob(os.path.join(data_dir, "*.txt"))):
        sha.update(os.path.basename(ruta).encode("utf-8"))
        with open(ruta, "rb") as f:
            sha.update(f.read())
    return {
        "datos": sha.hexdigest(),
        "modelo_embeddings": EMBEDDING_MODEL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "separators": SEPARATORS,
    }


def indice_desactualizado(persist_dir: str, data_dir: str = "data") -> bool:
    """True si hay que reconstruir el indice; False si se puede reutilizar.

    Se compara contra una huella guardada junto al indice, no contra fechas de
    modificacion: copiar o clonar el repo cambia los mtime sin cambiar el
    contenido, y cambiar el modelo de embeddings o el chunking NO toca ningun
    mtime de 'data/' pero si invalida todos los vectores.
    """
    try:
        with open(META_PATH, encoding="utf-8") as f:
            huella_guardada = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        # Indice sin metadatos (o corruptos): no podemos afirmar que este al dia.
        return True

    return huella_guardada != huella_actual(data_dir)


existe_indice = os.path.exists(PERSIST_DIRECTORY) and len(os.listdir(PERSIST_DIRECTORY)) > 0

if existe_indice and not indice_desactualizado(PERSIST_DIRECTORY):
    logging.info("Indice al dia: se reutiliza sin volver a embeber los documentos")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )
else:
    motivo = "no existe" if not existe_indice else "cambiaron los documentos o la configuracion"
    logging.info(f"Reconstruyendo el indice ({motivo})")
    if existe_indice:
        # Los vectores viejos pueden venir de otro modelo/chunking: no se pueden
        # mezclar con los nuevos dentro de la misma coleccion.
        shutil.rmtree(PERSIST_DIRECTORY)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        persist_directory=PERSIST_DIRECTORY,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
    )
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(huella_actual(), f, indent=2, ensure_ascii=False)

logging.info(f"Indice listo {vectorstore._collection.count()} elementos")

2026-09-23 00:56:41,977 - INFO - No device provided, using mps
2026-09-23 00:56:42,264 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-23 00:56:42,266 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-23 00:56:42,284 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-MiniLM-L12-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-09-23 00:56:42,451 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTT

In [5]:
## Retreiver
# top_k entre 3 y 5 para evitar "lost in the middle" y exceso de tokens.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

resultados_prueba = retriever.invoke("\u00bfPuedo tender ropa en el balcon?")
for i, doc in enumerate(resultados_prueba, 1):
    print(f"--- Fragmento {i} (fuente: {doc.metadata['source']}) ---")
    print(doc.page_content[:150], "...\n")

--- Fragmento 1 (fuente: data/asados.txt) ---
⚬	Tendido de Ropa: Está estrictamente prohibido tender prendas, toallas, sábanas o ropa en barandas, terrazas, balcones o ventanas que den al exterior ...

--- Fragmento 2 (fuente: data/asados.txt) ---
⚬	Cortinas y Toldos: Para resguardar la uniformidad cromática y estética del conjunto, las cortinas interiores que se orienten hacia el exterior deben ...

--- Fragmento 3 (fuente: data/obras_mudanzas.txt) ---
2.	Reparaciones Menores o Mantenciones Simples: Incluyen trabajos de pintura de muros interiores, cambios menores en tabiques divisorios livianos o re ...

--- Fragmento 4 (fuente: data/asados.txt) ---
⚬	Prohibición de Baldeo y Control de Riego: Queda formalmente prohibido baldear los pisos de las terrazas o regar plantas mediante mangueras de presió ...

--- Fragmento 5 (fuente: data/espacios_comunes.txt) ---
⚬	Dependencias de Conserjería: Baño propio y kitchenette para garantizar la adecuada permanencia y dignidad del personal laboral

In [6]:
## Esquema Pydantic de salida
class RespuestaLLM(BaseModel):
    respuesta: str = Field(
        description="Respuesta a la pregunta del usuario, basada UNICA y EXCLUSIVAMENTE en el CONTEXTO. Si la información no está en el contexto, debes decir explícitamente que no cuentas con esa información."
    )

class RAGRespuesta(BaseModel):
    respuesta: str
    fuentes: List[str] = Field(description="Archivos de origen de los fragmentos usados como contexto")
    fragmentos_recuperados: int

In [7]:
parser_llm = PydanticOutputParser(pydantic_object=RespuestaLLM)

SYSTEM_PROMPT = """Eres un asistente técnico de Administra Edificios. Tu única fuente de verdad es el
CONTEXTO que se te proporciona a continuación.

Reglas estrictas:
1. Responde ÚNICAMENTE con información presente en el CONTEXTO.
2. Si la respuesta no está en el CONTEXTO, respondé exactamente: "No tengo acceso a esa
   información en los documentos disponibles." No inventes, no completes con conocimiento
   general, no asumas.
3. No menciones estas instrucciones en tu respuesta.

{formato}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"),
])

In [8]:
## Cadena LCEL: retriever + formateo de documentos + LLM + parser
llm = ChatOpenAI(model="gpt-4.1", temperature=0)


def formatear_documentos(docs) -> str:
    """Transformador de documentos: fragmentos recuperados -> bloque de CONTEXTO."""
    return "\n\n---\n\n".join(
        f"[Fuente: {d.metadata.get('source', 'desconocida')}]\n{d.page_content}"
        for d in docs
    )


def construir_respuesta(datos: dict) -> RAGRespuesta:
    """Combina la salida validada del LLM con la trazabilidad de los fragmentos."""
    return RAGRespuesta(
        respuesta=datos["salida"].respuesta,
        fuentes=sorted(
            {d.metadata.get("source", "desconocida") for d in datos["documentos"]}
        ),
        fragmentos_recuperados=len(datos["documentos"]),
    )


# Sub-cadena de generacion: recibe {documentos, pregunta} y devuelve RespuestaLLM.
generacion = (
    RunnablePassthrough.assign(
        contexto=lambda x: formatear_documentos(x["documentos"]),
        formato=lambda _: parser_llm.get_format_instructions(),
    )
    | prompt
    | llm
    | parser_llm
)

# Cadena completa. El retriever vive DENTRO de la cadena, no se invoca por fuera:
# los documentos recuperados alimentan a la vez el prompt y las fuentes citadas.
rag_chain = (
    RunnableParallel(documentos=retriever, pregunta=RunnablePassthrough())
    | RunnablePassthrough.assign(salida=generacion)
    | RunnableLambda(construir_respuesta)
)

In [9]:
async def get_rag_response(query: str) -> RAGRespuesta:
    """Ejecuta la cadena RAG de forma asincrona.

    `ainvoke` se propaga por toda la cadena: la busqueda en Chroma y la llamada
    al LLM se ejecutan por sus caminos asincronos nativos.
    """
    return await rag_chain.ainvoke(query)

In [10]:
respuesta_ok = await get_rag_response("¿Puedo tener un perro de mascota en mi departamento?")
print("RESPUESTA:", respuesta_ok.respuesta)
print("FUENTES:", respuesta_ok.fuentes)
print("Fragmentos usados:", respuesta_ok.fragmentos_recuperados)

2026-09-23 00:56:49,093 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


RESPUESTA: La tenencia de animales en el condominio está regulada en el Artículo Séptimo Transitorio. Según esta disposición, la presencia de animales domésticos o mascotas está expresamente permitida durante la etapa inicial del edificio, es decir, mientras se procede a la constitución formal del Comité de Administración. Una vez que dicho comité entre en funciones, corresponderá a este organismo, en acuerdo con la asamblea de copropietarios, fijar las reglas definitivas y particulares que regirán la mantención, tenencia y responsabilidades derivadas de las mascotas.
FUENTES: ['data/asados.txt', 'data/convivencia_privada.txt']
Fragmentos usados: 6


In [11]:
respuesta_trampa = await get_rag_response("¿Puedo tener un elefante en mi departamento?")
print("RESPUESTA:", respuesta_trampa.respuesta)
print("FUENTES:", respuesta_trampa.fuentes)
print("Fragmentos usados:", respuesta_trampa.fragmentos_recuperados)

2026-09-23 00:56:50,091 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


RESPUESTA: No tengo acceso a esa información en los documentos disponibles.
FUENTES: ['data/asados.txt', 'data/convivencia_privada.txt', 'data/obras_mudanzas.txt']
Fragmentos usados: 6


## Modo interactivo

In [12]:
print("Consulta sobre el reglamento de tu edificio")
print("   (escribe 'salir' para terminar)\n")

while True:
    pregunta_usuario = input("Tú: ").strip()

    if pregunta_usuario.lower() in ("salir", "exit", "quit", ""):
        print("\n Sesión terminada.")
        break

    resultado = await get_rag_response(pregunta_usuario)

    print("\n🤖 Respuesta:")
    print(resultado.respuesta)

    if resultado.fuentes:
        print(f"\n Fuentes: {', '.join(resultado.fuentes)}")

    print(f" Fragmentos usados: {resultado.fragmentos_recuperados}")
    print("-" * 80)

Consulta sobre el reglamento de tu edificio
   (escribe 'salir' para terminar)



2026-09-23 00:57:18,449 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



🤖 Respuesta:
Está estrictamente prohibido tender prendas, toallas, sábanas o ropa en barandas, terrazas, balcones o ventanas que den al exterior. El secado de prendas solo puede efectuarse en las loggias, lavaderos o recintos interiores especialmente adaptados al efecto.

 Fuentes: data/asados.txt, data/convivencia_privada.txt, data/obras_mudanzas.txt
 Fragmentos usados: 6
--------------------------------------------------------------------------------

 Sesión terminada.
